In [2]:
from pretrain import pretrain
from finetune import finetune
from testing import evaluate, compute_final_results
from load_data import load_data
import json
import argparse

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

In [4]:
import os

def run(data_file,
                hyper_parameters,
                patient_list,
                unrestricted=True,
                combined=False,
                runs=1,
                specific_epochs=None,
                specific_epochs_pretrain=None,
                track_epochs=False,
                tensorboard=True,
                combo=False):

        # Check if the data file exists and is not empty
        if not os.path.exists(data_file):
            print(f"Error: Data file '{data_file}' not found.")
            return None
        if os.path.getsize(data_file) == 0:
            print(f"Error: Data file '{data_file}' is empty.")
            return None

        try:
            pretrain_data, scaling = load_data(data_file,
                                                              pretrain=True,
                                                              unrestricted=unrestricted,
                                                              combined=combined)
        except EOFError:
            print(f"Error loading data from '{data_file}': The file seems to be truncated or incomplete.")
            return None
        except UnpicklingError:
            print(f"Error loading data from '{data_file}': There was an error unpickling the data. The file might be corrupted.")
            return None


 #load_data takes input the pkl file. Outputs training, validation and testing set from it, as well as the scale value.

        pretrain(pretrain_data,
                         hyper_parameters,
                         unrestricted=unrestricted,
                         runs=runs,
                         specific_epochs=specific_epochs_pretrain,
                         track_epochs=track_epochs,
                         tensorboard=tensorboard,
                         combo=combo)

        finetune_data = {}
        scaling = {}
        for p in patient_list:
                (finetune_data[p], scaling[p]) = load_data(data_file,
                                                                                                   pretrain=False,
                                                                                                   patient=p,
                                                                                                   unrestricted=unrestricted,
                                                                                                   combined=combined)

        finetune(finetune_data,
                         hyper_parameters,
                         patient_list,
                         unrestricted=unrestricted,
                         runs=runs,
                         specific_epochs=specific_epochs,
                         track_epochs=track_epochs,
                         tensorboard=tensorboard)

        (results, best_models) = evaluate(finetune_data,
                                                                          patient_list,
                                                                          scaling,
                                                                          batch_size=hyper_parameters['batch_size'],
                                                                          testing_set='validation',
                                                                          unrestricted=unrestricted,
                                                                          runs=runs,
                                                                          tensorboard=tensorboard)

        final_results = compute_final_results(results, best_models, patient_list)

        return final_results

In [5]:
scenario= 'carbs'
runs= 10
case= 'unrestricted'
unrestricted= True
combined= False
fd = open(scenario + '_hyper_parameters_' + case + '.json', 'r')
hyper_parameters = json.load(fd)
fd.close()
batch_size = hyper_parameters['batch_size']
time= None



if scenario == 'carbs':
  patient_list = ['540', '544', '552', '559', '563', '575', '584', '588', '591', '596']
elif scenario == 'combo':
  patient_list = ['540', '544', '552', '559', '563', '575', '584', '588', '591']
else:
  patient_list = ['540', '552', '559', '563', '570', '575', '584', '588', '591', '596']

if scenario == 'combo':
  combo = True
else:
  combo = False

if time is None:
  data_file = scenario + '_data.pkl'
else:
  data_file = scenario + '_data' + str(time) + '.pkl'

if combined:
  track_epochs = False
  specify_epochs = True
else:
  track_epochs = True
  specify_epochs = False

if specify_epochs:
  fd = open('epochs.json', 'r')
  epochs = json.load(fd)
  fd.close()
  specific_epochs_pretrain = epochs['pretrain']
  specific_epochs = epochs['finetune']
else:
  specific_epochs_pretrain = None
  specific_epochs = None


In [6]:
display_results = run(data_file,
				  hyper_parameters,
				  patient_list,
				  runs=runs,
				  unrestricted=unrestricted,
				  combined=combined,
				  track_epochs=track_epochs,
				  specific_epochs=specific_epochs,
				  specific_epochs_pretrain=specific_epochs_pretrain,
				  combo=combo
				  )
fd = open('run_results.json', 'w')
json.dump(display_results, fd)
fd.close()

run 0
epch 200
EPOCH: 0
EPOCH: 1
EPOCH: 2
EPOCH: 3
EPOCH: 4
EPOCH: 5
EPOCH: 6
EPOCH: 7
EPOCH: 8
EPOCH: 9
EPOCH: 10
EPOCH: 11
EPOCH: 12
EPOCH: 13
EPOCH: 14
EPOCH: 15
EPOCH: 16
EPOCH: 17
EPOCH: 18
EPOCH: 19
EPOCH: 20
EPOCH: 21
EPOCH: 22
EPOCH: 23
EPOCH: 24
EPOCH: 25
run 1
epch 200
EPOCH: 0
EPOCH: 1
EPOCH: 2
EPOCH: 3
EPOCH: 4
EPOCH: 5
EPOCH: 6
EPOCH: 7
EPOCH: 8
EPOCH: 9
EPOCH: 10
EPOCH: 11
EPOCH: 12
EPOCH: 13
EPOCH: 14
EPOCH: 15
EPOCH: 16
EPOCH: 17
EPOCH: 18
EPOCH: 19
EPOCH: 20
EPOCH: 21
EPOCH: 22
EPOCH: 23
EPOCH: 24
EPOCH: 25
EPOCH: 26
EPOCH: 27
EPOCH: 28
EPOCH: 29
EPOCH: 30
EPOCH: 31
EPOCH: 32
EPOCH: 33
run 2
epch 200
EPOCH: 0
EPOCH: 1
EPOCH: 2
EPOCH: 3
EPOCH: 4
EPOCH: 5
EPOCH: 6
EPOCH: 7
EPOCH: 8
EPOCH: 9
EPOCH: 10
EPOCH: 11
EPOCH: 12
EPOCH: 13
EPOCH: 14
EPOCH: 15
EPOCH: 16
EPOCH: 17
EPOCH: 18
EPOCH: 19
EPOCH: 20
EPOCH: 21
EPOCH: 22
EPOCH: 23
EPOCH: 24
EPOCH: 25
EPOCH: 26
EPOCH: 27
EPOCH: 28
run 3
epch 200
EPOCH: 0
EPOCH: 1
EPOCH: 2
EPOCH: 3
EPOCH: 4
EPOCH: 5
EPOCH: 6
EPOCH: 7
EPOCH: 8

NameError: name 'data_file' is not defined